# Audit Hypothesis Agent — v2

RAG-агент для генерации аудиторских гипотез.

## Что исправлено по сравнению с v1:
- **Bug fix**: опечатка `normalize_embeddigns` → `normalize_embeddings` (запрос теперь нормализуется корректно)
- **Bug fix**: `add_generation_promt` → `add_generation_prompt` (chat-template работает правильно)
- **Bug fix**: дедупликация в `retrieve_with_rerank` перенесена за пределы цикла
- **Bug fix**: `dict_values` → `list` для корректной работы `zip`
- **Bug fix**: `rerunker` → `reranker` (единообразное именование)
- **Bug fix**: вложенные кавычки в f-string заменены на `'`
- **Feature**: добавлена реальная multi-query экспансия через LLM
- **Feature**: загрузка PDF и DOCX документов
- **Feature**: управление длиной контекста по токенам
- **Feature**: структурированный парсинг гипотез (Pydantic)
- **Refactor**: класс `AuditAgent` инкапсулирует логику вместо глобальных переменных
- **Refactor**: `DocumentLoader` — отдельный класс для загрузки документов
- **Security**: JSON-сериализация чанков вместо небезопасного pickle

## 1. Установка зависимостей

In [ ]:
%pip install -U sentence-transformers
%pip install python-docx mammoth pdfplumber
%pip install -U bitsandbytes
%pip install langchain_text_splitters faiss-cpu
%pip install pydantic
%pip install accelerate

# Опционально — docling даёт лучший парсинг сложных таблиц (PDF/DOCX → Markdown).
# Если установлен, DocumentLoader использует его автоматически.
# %pip install docling


In [ ]:
%pip install -q pyyaml

## 1.5 Конфигурация моделей

Редактируй `config.yaml` — ноутбук менять не нужно:
- Если папка `local_path` существует → используется локальная копия
- Если нет → модель скачивается с HuggingFace Hub автоматически
- `force_hub: true` → всегда скачивать (полезно для обновлений)

In [ ]:
import os
import yaml
from huggingface_hub import snapshot_download, login


def _hf_login():
    """
    Авторизация в HuggingFace Hub (опционально).
    Токен берётся из переменной окружения HF_TOKEN.
    Устанавливается один раз за сессию.

    Как задать токен (выбери один способ):
      1. Переменная окружения:  HF_TOKEN=hf_xxx  (рекомендуется)
      2. Файл .env в папке ноутбука:
             HF_TOKEN=hf_xxx
         затем %pip install python-dotenv и раскомментируй строки ниже.
      3. Напрямую в ячейке (только локально, НЕ коммитить!):
             os.environ['HF_TOKEN'] = 'hf_xxx'

    Получить токен: https://huggingface.co/settings/tokens
    """
    # Раскомментируй для загрузки из .env файла:
    # from dotenv import load_dotenv; load_dotenv()

    token = os.environ.get("HF_TOKEN")
    if token:
        login(token=token, add_to_git_credential=False)
        print("✓ HuggingFace: авторизация выполнена")
    else:
        print("ℹ HuggingFace: токен не задан (HF_TOKEN), используется анонимный доступ")
        print("  Публичные модели (Qwen, bge-m3, bge-reranker) доступны без токена.")


def resolve_model_path(local_path: str, hub_id: str, force_hub: bool = False) -> str:
    """
    Возвращает путь к модели:
      - Если local_path существует и force_hub=False → используем локальную копию.
      - Иначе → скачиваем с HuggingFace Hub в local_path через snapshot_download.
        При следующем запуске папка уже будет и скачивания не произойдёт.
    """
    if not force_hub and os.path.isdir(local_path):
        print(f"✓ Локальная модель найдена: {local_path}")
        return local_path

    if not force_hub:
        print(f"⚠ Папка '{local_path}' не найдена — скачиваем с Hub.")
    else:
        print(f"⬇ force_hub=True — принудительное скачивание.")

    print(f"⬇ Скачиваем {hub_id} → {local_path} ...")
    snapshot_download(
        repo_id=hub_id,
        local_dir=local_path,
        token=os.environ.get("HF_TOKEN"),  # None = анонимно
    )
    print(f"✓ Скачано в: {local_path}")
    return local_path


# Авторизация (если задан HF_TOKEN)
_hf_login()

# Загружаем конфигурацию
with open("config.yaml", "r", encoding="utf-8") as _f:
    _cfg = yaml.safe_load(_f)["models"]

model_path    = resolve_model_path(**_cfg["llm"])
embedder_path = resolve_model_path(**_cfg["embedder"])
reranker_path = resolve_model_path(**_cfg["reranker"])

print(f"\nLLM      : {model_path}")
print(f"Embedder : {embedder_path}")
print(f"Reranker : {reranker_path}")

## 2. Загрузка моделей

In [ ]:
import torch
from transformers import BitsAndBytesConfig

# Авто-определение устройства
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {device.upper()}")

if device == "cuda":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    load_dtype = torch.float16
    print("✓ GPU: 4-bit квантизация включена")
else:
    quant_config = None
    load_dtype = torch.bfloat16   # ~14 GB RAM для 7B (вместо ~28 GB в float32)
    print("ℹ CPU: квантизация отключена, используется bfloat16")
    print("  Для 7B модели нужно ~14 GB RAM. Для быстрого теста смени")
    print("  hub_id в config.yaml на Qwen/Qwen2.5-0.5B-Instruct (~1 GB)")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, CrossEncoder

if device == "cuda":
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("Загрузка моделей...")

# Пути берутся из config.yaml (ячейка выше)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map=device,
    quantization_config=quant_config,   # None на CPU
    torch_dtype=load_dtype,
)
print(f"Модель загружена на: {next(model.parameters()).device}")
if device == "cuda":
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# device берётся из ячейки выше (не захардкожен)
embedder = SentenceTransformer(embedder_path, device=device)
reranker  = CrossEncoder(reranker_path, device=device)

print("Модели загружены ✓")


## 3. Загрузчик документов (table-aware)

Парсит PDF / DOCX / TXT **с сохранением таблиц**:
- Таблицы → Markdown (привязка значений к столбцам сохраняется)
- Обёрнуты в маркеры `[ТАБЛИЦА]...[/ТАБЛИЦА]` — чанкер не режет их посередине
- **PDF**: `pdfplumber.extract_tables()` отдельно от текста
- **DOCX**: `python-docx` проходит абзацы и таблицы по порядку (`mammoth` таблицы терял!)
- **Опционально**: если установлен `docling` — используется он (лучше на сложных таблицах)


In [ ]:
import os
import pdfplumber
from typing import Optional, List, Iterator


# ── Опциональная проверка наличия docling ────────────────────────────
try:
    from docling.document_converter import DocumentConverter
    _HAS_DOCLING = True
except Exception:
    _HAS_DOCLING = False


def _table_to_markdown(rows: List[List]) -> str:
    """
    Преобразует таблицу (список строк) в Markdown-таблицу.
    Markdown сохраняет привязку значений к столбцам — критично для
    финансовых таблиц (суммы, лимиты, проводки).
    """
    if not rows:
        return ""
    # Нормализуем: None → "", убираем переносы строк внутри ячеек
    clean = [
        [(str(c) if c is not None else "").replace("\n", " ").strip() for c in row]
        for row in rows
    ]
    ncols = max(len(r) for r in clean)
    clean = [r + [""] * (ncols - len(r)) for r in clean]  # выравниваем ширину

    header = clean[0]
    lines = ["| " + " | ".join(header) + " |"]
    lines.append("| " + " | ".join(["---"] * ncols) + " |")
    for row in clean[1:]:
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines)


def _wrap_table(md_table: str) -> str:
    """Оборачивает таблицу в маркеры — помогает чанкеру и LLM распознать её."""
    return f"\n\n[ТАБЛИЦА]\n{md_table}\n[/ТАБЛИЦА]\n\n"


class DocumentLoader:
    """
    Загружает текст из PDF, DOCX, TXT с сохранением таблиц.

    Таблицы конвертируются в Markdown и оборачиваются в [ТАБЛИЦА]...[/ТАБЛИЦА].
    Если установлен docling — используется он (лучше на сложных таблицах),
    иначе fallback на pdfplumber (PDF) и python-docx (DOCX).
    """

    @staticmethod
    def load(file_path: str) -> str:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Файл не найден: {file_path}")

        ext = os.path.splitext(file_path)[1].lower()

        # docling умеет PDF и DOCX — пробуем его первым
        if _HAS_DOCLING and ext in (".pdf", ".docx"):
            try:
                return DocumentLoader._load_docling(file_path)
            except Exception as e:
                print(f"  docling не справился ({e}), fallback на встроенный парсер")

        if ext == ".pdf":
            return DocumentLoader._load_pdf(file_path)
        elif ext in (".docx", ".doc"):
            return DocumentLoader._load_docx(file_path)
        elif ext == ".txt":
            with open(file_path, "r", encoding="utf-8") as f:
                return f.read()
        else:
            raise ValueError(f"Неподдерживаемый формат: {ext}")

    # ── docling (опционально) ────────────────────────────────────────
    @staticmethod
    def _load_docling(file_path: str) -> str:
        """docling: layout-aware парсинг, таблицы → Markdown «из коробки»."""
        converter = DocumentConverter()
        result = converter.convert(file_path)
        md = result.document.export_to_markdown()
        print(f"docling загрузил: {len(md)} символов (Markdown с таблицами)")
        return md

    # ── PDF: текст + таблицы отдельно ────────────────────────────────
    @staticmethod
    def _load_pdf(file_path: str) -> str:
        """
        Извлекает текст постранично и отдельно — таблицы как Markdown.
        Таблицы добавляются в конец страницы в маркерах [ТАБЛИЦА].
        """
        parts = []
        n_tables = 0
        with pdfplumber.open(file_path) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                text = page.extract_text() or ""
                if text:
                    parts.append(text)
                # Таблицы извлекаем отдельным методом — сохраняет структуру
                for table in page.extract_tables():
                    if table and len(table) > 1:
                        parts.append(_wrap_table(_table_to_markdown(table)))
                        n_tables += 1
        result = "\n\n".join(parts)
        print(f"PDF загружен: {len(pdf.pages)} стр., {n_tables} таблиц, {len(result)} символов")
        return result

    # ── DOCX: python-docx сохраняет таблицы (mammoth их терял!) ──────
    @staticmethod
    def _load_docx(file_path: str) -> str:
        """
        Парсит DOCX через python-docx, проходя абзацы и таблицы
        в порядке их следования в документе.
        """
        from docx import Document
        from docx.table import Table
        from docx.text.paragraph import Paragraph
        from docx.oxml.ns import qn

        doc = Document(file_path)
        parts = []
        n_tables = 0

        # Итерируем блоки (абзацы и таблицы) в порядке документа
        body = doc.element.body
        for child in body.iterchildren():
            if child.tag == qn("w:p"):
                para = Paragraph(child, doc)
                if para.text.strip():
                    parts.append(para.text)
            elif child.tag == qn("w:tbl"):
                table = Table(child, doc)
                rows = [[cell.text for cell in row.cells] for row in table.rows]
                if rows and len(rows) > 1:
                    parts.append(_wrap_table(_table_to_markdown(rows)))
                    n_tables += 1

        result = "\n\n".join(parts)
        print(f"DOCX загружен: {n_tables} таблиц, {len(result)} символов")
        return result


## 4. Ретривер

**Изменения vs v1:**
- `normalize_embeddings` (опечатка исправлена)
- Хранение через JSON вместо pickle (безопасность)
- Опциональный HNSW-индекс для больших коллекций
- `add_document` принимает путь к файлу напрямую

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
from typing import List, Dict
import faiss
import json
import re


class DocumentRetriever:
    def __init__(self, embedder, chunk_size: int = 1200, chunk_overlap: int = 200,
                 index_dir: str = ".", use_hnsw: bool = False):
        """
        Args:
            embedder: SentenceTransformer-совместимый энкодер.
            chunk_size: Размер чанка в символах.
                        1200 ≈ 300-400 токенов для русского текста —
                        оптимально для аудиторских/регуляторных документов.
            chunk_overlap: Перекрытие чанков (~16% от chunk_size).
                           Гарантирует, что граничные предложения попадут
                           хотя бы в один полный чанк.
            index_dir: Директория для сохранения индекса.
            use_hnsw: Использовать HNSW (быстрее для >50k векторов).
        """
        self.embedder = embedder
        self.chunks: List[Dict] = []
        self.index = None
        self.dimension: Optional[int] = None
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.use_hnsw = use_hnsw

        self._index_path = os.path.join(index_dir, "index.faiss")
        self._chunks_path = os.path.join(index_dir, "chunks.json")

        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[
                "[/ТАБЛИЦА]",  # граница таблицы — высший приоритет (не резать таблицу)
                "[ТАБЛИЦА]",
                "\n\n",    # разрыв абзаца
                ".\n",      # конец предложения + перенос строки (частый в НПА)
                "\n",       # перенос строки
                ". ",        # конец предложения
                "; ",        # точка с запятой (перечисления в регуляторных текстах)
                ", ",
                " ",
                "",
            ],
            length_function=len,
            is_separator_regex=False,
        )
        self._load_existing_db()

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def add_file(self, file_path: str):
        """Загружает файл и добавляет его в БД."""
        text = DocumentLoader.load(file_path)
        metadata = {"source": os.path.basename(file_path)}
        self.add_document(text, metadata)

    def add_document(self, text: str, metadata: Optional[Dict] = None):
        """
        Добавляет текст в БД.
        Каждый чанк получает в metadata заголовок ближайшего раздела —
        это позволяет LLM понимать контекст (из какого раздела документа фрагмент).
        """
        print("Разбиваем текст на чанки...")
        raw_chunks = self.text_splitter.split_text(text)

        if not raw_chunks:
            print("Нет чанков для добавления")
            return

        # Определяем заголовок раздела для каждого чанка
        section_map = self._build_section_map(text)
        enriched = []
        for chunk in raw_chunks:
            pos = text.find(chunk[:60])     # позиция чанка в исходном тексте
            section = self._find_section(section_map, pos)
            # Добавляем заголовок раздела в начало чанка —
            # эмбеддинг и LLM получают явный контекст раздела
            headed = f"[{section}]\n{chunk}" if section else chunk
            chunk_meta = dict(metadata or {})
            chunk_meta["section"] = section or ""
            enriched.append((headed, chunk_meta))

        texts_to_embed = [t for t, _ in enriched]
        print(f"Создание эмбеддингов для {len(texts_to_embed)} чанков...")
        new_embeddings = self.embedder.encode(
            texts_to_embed,
            batch_size=32,
            normalize_embeddings=True,
            show_progress_bar=True,
        ).astype("float32")

        if self.index is None:
            self.dimension = new_embeddings.shape[1]
            self.index = self._create_index(self.dimension)
            print(f"Создан новый FAISS индекс (dim={self.dimension}, hnsw={self.use_hnsw})")

        self.index.add(new_embeddings)

        for text_chunk, meta in enriched:
            self.chunks.append({"text": text_chunk, "metadata": meta})

        print(f"Добавлено {len(enriched)} чанков. Всего в БД: {len(self.chunks)}")
        self._save_db()

    def search(self, query: str, top_k: int = 10) -> List[Dict]:
        """Ищет top_k релевантных чанков по запросу."""
        if self.index is None or self.index.ntotal == 0:
            print("БД пуста. Добавьте документы через add_document() или add_file().")
            return []

        query_embedding = self.embedder.encode(
            [query],
            normalize_embeddings=True,
        ).astype("float32")

        k = min(top_k, self.index.ntotal)
        scores, indices = self.index.search(query_embedding, k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx != -1 and idx < len(self.chunks):
                results.append({
                    "text": self.chunks[idx]["text"],
                    "metadata": self.chunks[idx]["metadata"],
                    "similarity": float(score),
                })
        return results

    def get_stats(self) -> Dict:
        return {
            "total_chunks": len(self.chunks),
            "total_vectors": self.index.ntotal if self.index else 0,
            "dimension": self.dimension,
            "chunk_size": self.chunk_size,
            "chunk_overlap": self.chunk_overlap,
            "index_type": "HNSW" if self.use_hnsw else "FlatIP",
        }

    # ------------------------------------------------------------------
    # Section-aware helpers
    # ------------------------------------------------------------------

    # Паттерны заголовков в аудиторских/регуляторных документах (НПА, положения, инструкции)
    _HEADER_RE = re.compile(
        r"^(?:"
        r"(?:\d+\.)+\s+[А-ЯЁA-Z][^\n]{3,60}"   # 1.2.3 Заголовок
        r"|(?:Раздел|Глава|Статья|Пункт|Приложение)\s+[\dА-ЯЁ][^\n]{0,60}"
        r"|[А-ЯЁ]{3,}[^\n]{0,60}"                  # ПОЛНОСТЬЮ ЗАГЛАВНЫЕ заголовки
        r")",
        re.MULTILINE,
    )

    def _build_section_map(self, text: str) -> List[tuple]:
        """Возвращает список (позиция, заголовок) для всех найденных заголовков."""
        return [(m.start(), m.group().strip()) for m in self._HEADER_RE.finditer(text)]

    def _find_section(self, section_map: List[tuple], pos: int) -> Optional[str]:
        """Находит ближайший заголовок, предшествующий позиции pos."""
        current = None
        for start, title in section_map:
            if start <= pos:
                current = title
            else:
                break
        return current

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _create_index(self, dim: int):
        if self.use_hnsw:
            index = faiss.IndexHNSWFlat(dim, 32)
            index.hnsw.efConstruction = 200
            index.hnsw.efSearch = 64
            return index
        return faiss.IndexFlatIP(dim)

    def _load_existing_db(self):
        """Загружает сохранённую БД."""
        if not os.path.exists(self._index_path) or not os.path.exists(self._chunks_path):
            print("Сохранённая БД не найдена. Будет создана новая.")
            return False
        try:
            self.index = faiss.read_index(self._index_path)
            self.dimension = self.index.d
            with open(self._chunks_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                self.chunks = data["chunks"]
                self.chunk_size = data.get("chunk_size", self.chunk_size)
                self.chunk_overlap = data.get("chunk_overlap", self.chunk_overlap)
            print(f"БД загружена: {self.index.ntotal} векторов, {len(self.chunks)} чанков")
            return True
        except Exception as e:
            print(f"Ошибка загрузки БД: {e}")
            return False

    def _save_db(self):
        """Сохраняет БД на диск."""
        if self.index is None or self.index.ntotal == 0:
            return
        faiss.write_index(self.index, self._index_path)
        with open(self._chunks_path, "w", encoding="utf-8") as f:
            json.dump(
                {"chunks": self.chunks, "chunk_size": self.chunk_size, "chunk_overlap": self.chunk_overlap},
                f, ensure_ascii=False, indent=2,
            )
        print(f"БД сохранена: {self.index.ntotal} векторов, {len(self.chunks)} чанков")


## 5. Multi-query + Reranking

**Изменения vs v1:**
- `expand_query` теперь реально генерирует альтернативные формулировки через LLM
- дедупликация вынесена за пределы цикла
- `dict_values` явно конвертируется в `list` перед `zip`

In [ ]:
def expand_query(question: str, model, tokenizer, n_variants: int = 3) -> List[str]:
    """
    Генерирует n_variants перефразировок вопроса для multi-query retrieval.
    Улучшает recall, особенно для коротких запросов.
    """
    prompt = (
        f"Сгенерируй {n_variants} разных формулировки следующего аудиторского вопроса. "
        f"Каждую формулировку напиши на новой строке, без нумерации и пояснений.\n\n"
        f"Вопрос: {question}\n\nФормулировки:"
    )
    messages = [
        {"role": "system", "content": "Ты — помощник аудитора. Перефразируй запросы точно и кратко."},
        {"role": "user", "content": prompt},
    ]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,   # FIX: было add_generation_promt
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,    # низкая температура для более детерминированных перефразировок
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(generated_ids, skip_special_tokens=True)
    variants = [line.strip() for line in raw.splitlines() if line.strip()]
    # Всегда включаем оригинальный вопрос первым
    queries = [question] + variants[:n_variants]
    print(f"Multi-query: {len(queries)} формулировок")
    return queries


def retrieve_with_rerank(question: str, retriever: DocumentRetriever,
                         reranker: CrossEncoder,
                         model=None, tokenizer=None,
                         top_k: int = 8,
                         use_multi_query: bool = True) -> List[Dict]:
    """
    1. (опционально) расширяет запрос через LLM
    2. ищет по всем формулировкам
    3. дедуплицирует
    4. ранжирует через CrossEncoder
    """
    if use_multi_query and model is not None and tokenizer is not None:
        queries = expand_query(question, model, tokenizer)
    else:
        queries = [question]

    # Сбор результатов по всем запросам
    all_results = []
    for q in queries:
        all_results.extend(retriever.search(q, top_k=top_k))

    # FIX: дедупликация вынесена за пределы цикла
    unique_map = {r["text"]: r for r in all_results}
    unique_list = list(unique_map.values())   # FIX: явный list() для корректной работы zip

    if not unique_list:
        return []

    texts = [r["text"] for r in unique_list]
    pairs = [[question, t] for t in texts]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(unique_list, scores),   # FIX: list уже материализован
        key=lambda x: x[1],
        reverse=True,
    )
    return [r for r, _score in ranked[:top_k]]

## 6. Структурированный вывод (Pydantic)

Парсинг ответа LLM в типизированные объекты — упрощает downstream-обработку.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, Literal
import re

RISK_EMOJI = {"HIGH": "🔴", "MEDIUM": "🟡", "LOW": "🟢"}


class AuditHypothesis(BaseModel):
    """Одна аудиторская гипотеза."""
    number: int
    title: str = Field(description="Краткое описание гипотезы")
    risk_level: Literal["HIGH", "MEDIUM", "LOW"] = Field(
        default="MEDIUM",
        description="Уровень риска: HIGH / MEDIUM / LOW",
    )
    rationale: str = Field(description="Обоснование")
    consequences: str = Field(description="Возможные последствия")
    verification_steps: str = Field(description="Шаги проверки")


class HypothesesReport(BaseModel):
    """Полный отчёт с гипотезами."""
    question: str
    hypotheses: List[AuditHypothesis]
    sources_used: int
    source_names: List[str] = Field(default_factory=list)

    def to_markdown(self) -> str:
        lines = ["## Аудиторские гипотезы", f"**Вопрос:** {self.question}", ""]
        for h in self.hypotheses:
            emoji = RISK_EMOJI.get(h.risk_level, "")
            lines.append(f"### Гипотеза {h.number} {emoji} [{h.risk_level}]: {h.title}")
            lines.append(f"- **Обоснование:** {h.rationale}")
            lines.append(f"- **Последствия:** {h.consequences}")
            lines.append(f"- **Проверка:** {h.verification_steps}")
            lines.append("")
        if self.source_names:
            lines.append("**Источники:**")
            for s in self.source_names:
                lines.append(f"  - {s}")
        else:
            lines.append(f"*Использовано фрагментов: {self.sources_used}*")
        return "\n".join(lines)


def parse_hypotheses(raw_answer: str, question: str,
                     chunks: List[Dict]) -> "HypothesesReport":
    """
    Парсит ответ LLM в HypothesesReport.
    Если формат не распознан — возвращает одну гипотезу с полным текстом (fallback).
    """
    hypotheses = []
    blocks = re.split(r"\n(?=\d+\.\s+(?:Гипотеза|Hypothesis))", raw_answer.strip())

    for i, block in enumerate(blocks, 1):
        if not block.strip():
            continue
        title_match = re.search(r"^\d+\.\s+(?:Гипотеза \d+:\s*)?(.+)", block, re.MULTILINE)
        title = title_match.group(1).strip() if title_match else f"Гипотеза {i}"

        risk_raw = _extract_section(block, ["Уровень риска", "Риск", "Risk"]) or ""
        risk_level = "HIGH" if "HIGH" in risk_raw.upper() or "ВЫСОК" in risk_raw.upper() \
            else "LOW" if "LOW" in risk_raw.upper() or "НИЗК" in risk_raw.upper() \
            else "MEDIUM"

        hypotheses.append(AuditHypothesis(
            number=i,
            title=title,
            risk_level=risk_level,
            rationale=_extract_section(block, ["Обоснование"]) or "—",
            consequences=_extract_section(block, ["Последствия"]) or "—",
            verification_steps=_extract_section(block, ["Проверка", "Шаги проверки"]) or "—",
        ))

    # Fallback: если ничего не распознано — оборачиваем весь ответ
    if not hypotheses:
        hypotheses = [AuditHypothesis(
            number=1,
            title="Анализ (формат не распознан)",
            risk_level="MEDIUM",
            rationale=raw_answer[:1000],
            consequences="—",
            verification_steps="Требуется ручная обработка ответа модели",
        )]

    # Уникальные источники из метаданных чанков
    seen, source_names = set(), []
    for c in chunks:
        meta = c.get("metadata", {})
        src = meta.get("source", "")
        sec = meta.get("section", "")
        label = f"{src} / {sec}" if sec else src
        if label and label not in seen:
            seen.add(label)
            source_names.append(label)

    return HypothesesReport(
        question=question,
        hypotheses=hypotheses,
        sources_used=len(chunks),
        source_names=source_names,
    )


def _extract_section(text: str, labels: List[str]) -> Optional[str]:
    for label in labels:
        pattern = rf"-?\s*\*?\*?{re.escape(label)}\*?\*?:?\s*(.+?)(?=\n\s*-|\n\s*\*|$)"
        m = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return None


## 7. Основной агент

Класс `AuditAgent` инкапсулирует все компоненты вместо глобальных переменных.

In [ ]:
class AuditAgent:
    """
    RAG-агент для генерации аудиторских гипотез.

    Инкапсулирует retriever, model, tokenizer, reranker.
    """

    # Максимальная длина контекста в символах (~75% от лимита модели)
    MAX_CONTEXT_CHARS = 12_000

    def __init__(self, model, tokenizer, embedder, reranker,
                 chunk_size: int = 800, chunk_overlap: int = 100,
                 index_dir: str = "."):
        self.model = model
        self.tokenizer = tokenizer
        self.reranker = reranker
        self.retriever = DocumentRetriever(
            embedder=embedder,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            index_dir=index_dir,
        )

    def add_file(self, file_path: str):
        """Добавляет документ в базу знаний."""
        self.retriever.add_file(file_path)

    def ask(self, question: str, top_k: int = 8,
            use_multi_query: bool = True) -> HypothesesReport:
        """
        Главный метод: принимает вопрос, возвращает структурированный отчёт.

        Args:
            question: Аудиторский вопрос на русском языке.
            top_k: Количество чанков для контекста.
            use_multi_query: Включить расширение запроса через LLM.
        """
        print("[1/3] Поиск релевантных фрагментов...")
        chunks = retrieve_with_rerank(
            question, self.retriever, self.reranker,
            model=self.model if use_multi_query else None,
            tokenizer=self.tokenizer if use_multi_query else None,
            top_k=top_k,
            use_multi_query=use_multi_query,
        )

        if not chunks:
            print("Предупреждение: база знаний пуста. Гипотезы будут сгенерированы без контекста.")

        context = self._build_context(chunks)
        prompt = self._build_prompt(context, question)

        print("[2/3] Генерация гипотез...")
        raw_answer = self._generate(prompt)

        print("[3/3] Парсинг результатов...")
        report = parse_hypotheses(raw_answer, question, chunks)

        return report

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _build_context(self, chunks: List[Dict]) -> str:
        """Строит контекст с ограничением по длине."""
        parts = []
        total = 0
        for i, chunk in enumerate(chunks):
            meta = chunk.get("metadata", {})
            src  = meta.get("source", "")
            sec  = meta.get("section", "")
            # Формируем заголовок: [ИСТОЧНИК N | файл | раздел]
            header_parts = [f"ИСТОЧНИК {i + 1}"]
            if src:
                header_parts.append(src)
            if sec:
                header_parts.append(sec)
            header = "[" + " | ".join(header_parts) + "]"
            part = f"{header}\n{chunk['text']}"
            if total + len(part) > self.MAX_CONTEXT_CHARS:
                print(f"Контекст обрезан до {i} чанков (лимит {self.MAX_CONTEXT_CHARS} символов)")
                break
            parts.append(part)
            total += len(part)
        return "\n\n---\n\n".join(parts)

    def _build_prompt(self, context: str, question: str) -> str:
        return f"""Ты — AI-агент для генерации аудиторских гипотез. На основе предоставленных документов предложи 3-5 обоснованных гипотез для дальнейшего аудиторского анализа.

Каждая гипотеза должна содержать:
- краткое описание сути;
- обоснование (почему актуальна, на какие риски указывает);
- возможные последствия;
- уровень риска: HIGH / MEDIUM / LOW;
- шаги для проверки.

Формат ответа:
1. Гипотеза 1: [описание]
   - Уровень риска: HIGH / MEDIUM / LOW
   - Обоснование: [аргументы]
   - Последствия: [кратко]
   - Проверка: [шаги]
2. Гипотеза 2:
   ...

Критерии: гипотезы должны быть конкретными, релевантными и проверяемыми.

Контекст для анализа:
{context}

Вопрос:
{question}
"""

    def _generate(self, prompt: str) -> str:
        """Генерирует ответ через LLM."""
        messages = [
            {"role": "system", "content": "Ты — ассистент ведущего аудитора банковской группы."},
            {"role": "user", "content": prompt},
        ]
        formatted_prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,   # FIX: было add_generation_promt
        )
        inputs = self.tokenizer(formatted_prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=2048,
                temperature=0.6,   # чуть ниже для более детерминированных гипотез
                top_p=0.9,
            repetition_penalty=1.1,
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(generated_ids, skip_special_tokens=True)

## 8. Инициализация агента

In [ ]:
agent = AuditAgent(
    model=model,
    tokenizer=tokenizer,
    embedder=embedder,
    reranker=reranker,
    chunk_size=1200,    # было 800 — теперь соответствует улучшённым дефолтам DocumentRetriever
    chunk_overlap=200,  # было 100
    index_dir=".",
)

print("Агент готов.")
print("Статистика базы знаний:", agent.retriever.get_stats())


## 9. Загрузка документов (пример)

In [ ]:
# Пример загрузки документов:
# agent.add_file("documents/policy_cards.pdf")
# agent.add_file("documents/transactions_2024.docx")
# agent.add_file("documents/internal_rules.txt")

print("Загрузите документы через agent.add_file('путь/к/файлу')")

## 10. Запрос к агенту

In [ ]:
question = (
    "Вычитка информации о финансовых транзакциях по бизнес-карте "
    "корпоративного клиента и их отображение по счёту клиента"
)

report = agent.ask(question, top_k=8, use_multi_query=True)

print("\n" + "=" * 60)
print(report.to_markdown())
print("=" * 60)
print(f"Гипотез сгенерировано : {len(report.hypotheses)}")
print(f"Фрагментов использовано: {report.sources_used}")
if report.source_names:
    print("Источники:")
    for s in report.source_names:
        print(f"  • {s}")
